In [8]:
# Assignment 01
from pyspark.sql import SparkSession
import json

 
spark = SparkSession.builder \
    .appName("Generate Parquet Data") \
    .config("spark.sql.parquet.enableStats", "true") \
    .getOrCreate()

dict = {"num_rows": "", "format": "", "status": ""}

df = spark.read.parquet("/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/FirstProject/src/leetcode/large_practice_data.parquet")

df.show(5)

if df is not None:
    dict["num_rows"] = df.count()
    
    # Get the source path from the Spark plan
    logical_plan = df._jdf.queryExecution().logical()
        
    source_info = logical_plan.toString()

    if "Parquet" in source_info:
        file_format = "parquet"
    elif "CSV" in source_info:
        file_format = "csv"
    else:
        file_format = "unknown"
    print(f"Detected Format: {file_format}")
    dict["format"] = file_format

    dict["status"] = "success"



print (dict)

#create a json file to store the metadata

with open("/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/Feb_2026/metadata.json", "w") as f:
    json.dump(dict, f, indent=4)


 
spark.stop()

+---+--------------+-------+-----------+--------------------+-----+----------------+------------------+------------------+-----------+-----------------+---------------+---------------+---------------+
| id|transaction_id|user_id|   category|           timestamp| name|           email|            amount|             score|category_id|         discount|date_mm_dd_yyyy|date_dd_mm_yyyy|date_yyyy_mm_dd|
+---+--------------+-------+-----------+--------------------+-----+----------------+------------------+------------------+-----------+-----------------+---------------+---------------+---------------+
|900|           900|   4853|       null|2023-02-13 15:35:...|Alice|            null| 376.2585170603643|              null|       null|             null|     02-13-2023|           null|           null|
|901|           901|   5779|Electronics|2023-08-25 04:44:...|Alice|            null|105.43840119496156| 42.47771204426548|          8|             null|     08-25-2023|     25-08-2023|           n

In [ ]:
# Assignment 02
''' This should read the record.json file along with input file .
    If the status in record.json is valid then perform the below activity else exit the program with exception as "Data is not correct" 
    The Names should be capitalised .
    The date should be uniform ( DD-MM-YYYY)
    Null should be replaced with Unknown for text field , 0 for numeric fields.'''

import json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType, IntegerType, FloatType, DoubleType,
    DateType, TimestampType, BooleanType
)


spark = SparkSession.builder.appName("Read Parquet Data").getOrCreate()


with open("/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/Feb_2026/metadata.json", "r") as f:
    metadata = json.load(f)

if metadata["status"] == "success":
    
    df = spark.read.parquet(
        "/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/FirstProject/src/leetcode/large_practice_data.parquet"
    )
    df.show(10, truncate=False)
    
    for field in df.schema.fields:
        col_name = field.name
        dtype = field.dataType

       
        if isinstance(dtype, StringType):
            df = df.withColumn(col_name, F.initcap(F.col(col_name)))
        elif isinstance(dtype, (DateType, TimestampType)):
            df = df.withColumn(col_name, F.date_format(F.col(col_name), "dd-MM-yyyy"))

    for field in df.schema.fields:
        col_name = field.name
        dtype = field.dataType

        if isinstance(dtype, StringType):
            df = df.withColumn(col_name, F.coalesce(F.col(col_name), F.lit("Unknown")))

        elif isinstance(dtype, (IntegerType, FloatType, DoubleType)):
            df = df.withColumn(col_name, F.coalesce(F.col(col_name), F.lit(0)))

        elif isinstance(dtype, BooleanType):
            df = df.withColumn(col_name, F.coalesce(F.col(col_name), F.lit(False)))

  
    df.show(10, truncate=False)
else:
    raise Exception("Data is not correct")


spark.stop()


'''
            if "date" in col_name.lower():
                df = df.withColumn(
                    col_name,
                    F.date_format(
                        F.coalesce(
                            F.to_date(F.col(col_name), "MM-dd-yyyy"),
                            F.to_date(F.col(col_name), "dd-MM-yyyy"),
                            F.to_date(F.col(col_name), "yyyy-MM-dd")
                        ),
                        "dd-MM-yyyy"
                    )
                )
            elif "name" in col_name.lower():'''
       

+---+--------------+-------+-----------+--------------------------+-----+----------------+------------------+------------------+-----------+------------------+---------------+---------------+---------------+
|id |transaction_id|user_id|category   |timestamp                 |name |email           |amount            |score             |category_id|discount          |date_mm_dd_yyyy|date_dd_mm_yyyy|date_yyyy_mm_dd|
+---+--------------+-------+-----------+--------------------------+-----+----------------+------------------+------------------+-----------+------------------+---------------+---------------+---------------+
|900|900           |4853   |null       |2023-02-13 15:35:59.919691|Alice|null            |376.2585170603643 |null              |null       |null              |02-13-2023     |null           |null           |
|901|901           |5779   |Electronics|2023-08-25 04:44:40.928345|Alice|null            |105.43840119496156|42.47771204426548 |8          |null              |08-25-202

'\n            if "date" in col_name.lower():\n                df = df.withColumn(\n                    col_name,\n                    F.date_format(\n                        F.coalesce(\n                            F.to_date(F.col(col_name), "MM-dd-yyyy"),\n                            F.to_date(F.col(col_name), "dd-MM-yyyy"),\n                            F.to_date(F.col(col_name), "yyyy-MM-dd")\n                        ),\n                        "dd-MM-yyyy"\n                    )\n                )\n            elif "name" in col_name.lower():'

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("Trasnformations and aggregations using above data").getOrCreate()

df1 = spark.read.parquet(
    "/Users/yash/Desktop/MyProjects/cloudprojects/dataeng/FirstProject/src/leetcode/large_practice_data.parquet"
)

df1.show(10, truncate=False)
df1.printSchema()

df1 = df1.fillna({"name": "Unknown", "amount": 0, "category": "default_category"})

# amount > 0 
df2 = df1.filter(F.col("amount") > 0)

df2.show(10, truncate=False)

# replace null values in the "name" column with "Unknown"
df3 = df2.withColumn("name", F.coalesce(F.col("name"), F.lit("Unknown")))

df3.show(10, truncate=False)

df4 = df3.groupby(F.col("category")).agg(F.avg("amount").alias("average_amount"))

df4.show(10, truncate=False)
    
df5 = df3.groupby(F.col("category")).agg(F.countDistinct(F.col("email")).alias("unique_emails"), F.sum("amount").alias("total_amount"))

df5.show(10, truncate=False)

df3.groupby(F.col("category")).agg(F.count("email").alias("email_count")).show(10, truncate=False)

df3.select("email").show(10, truncate=False)

df3.dropDuplicates(["email"]).show(10, truncate=False)



spark.stop()

+---+--------------+-------+-----------+--------------------------+-----+----------------+------------------+------------------+-----------+------------------+---------------+---------------+---------------+
|id |transaction_id|user_id|category   |timestamp                 |name |email           |amount            |score             |category_id|discount          |date_mm_dd_yyyy|date_dd_mm_yyyy|date_yyyy_mm_dd|
+---+--------------+-------+-----------+--------------------------+-----+----------------+------------------+------------------+-----------+------------------+---------------+---------------+---------------+
|900|900           |4853   |null       |2023-02-13 15:35:59.919691|Alice|null            |376.2585170603643 |null              |null       |null              |02-13-2023     |null           |null           |
|901|901           |5779   |Electronics|2023-08-25 04:44:40.928345|Alice|null            |105.43840119496156|42.47771204426548 |8          |null              |08-25-202